In [31]:
# -*- coding: utf-8 -*-
"""
CatBoost 信贷违约预测 — v8a（自动剔除可疑列）
-----------------------------------------
在 v8 的基础上：
- 训练前自动读取两类黑名单：
  1) output_GPT2/auto_drop_features_v8a.txt （由 suspect_report + 重要性规则生成）
  2) output_GPT2/manual_drop_features.txt    （你手工维护，优先级同样高）
- 仅保留 days_since_* / stmt_active_days 等无窗口时间派生；无 30/60/90d 窗口。
- 其它流程与 v8 保持一致（无泄漏 winsorize、稳健CV、Bernoulli 采样）。
"""

import os, re, warnings, math
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")

BASE = 'output_GPT2'
os.makedirs(BASE, exist_ok=True)

SEEDS = [42, 3407]
np.random.seed(42)

WINSOR_LO = 0.01
WINSOR_HI = 0.995
N_SPLITS = 5
EMBARGO_FRAC = 0.02

def load_csv_safe(path):
    if not os.path.exists(path):
        return None
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception:
        for enc in ["utf-8","gbk","latin1"]:
            try:
                return pd.read_csv(path, low_memory=False, encoding=enc)
            except Exception:
                continue
        raise

def infer_id_col(df):
    cands = [c for c in df.columns if c.lower() in ["id","loan_id","user_id","customer_id","cust_id","account_id","client_id"]]
    if cands: return cands[0]
    for c in df.columns:
        if re.fullmatch(r".*id$", c, flags=re.I): return c
    return df.columns[0]

def infer_target_col(df):
    preferred = ["target","label","is_default","default","bad","y"]
    for name in preferred:
        for c in df.columns:
            if c.lower()==name: return c
    for c in df.columns:
        vals = df[c].dropna().unique()
        if len(vals)==2 and set(vals) <= {0,1}: return c
    return None

def ensure_binary_labels(y):
    if set(pd.unique(y.dropna())) <= {0,1}: return y.astype(int)
    mapping = {'y':1,'yes':1,'true':1,'bad':1,'default':1,'违约':1,'是':1,
               'n':0,'no':0,'false':0,'good':0,'nondefault':0,'否':0}
    yy = y.map(lambda v: mapping.get(str(v).strip().lower(), np.nan))
    if yy.dropna().nunique()==2: return yy.fillna(0).astype(int)
    raise ValueError("目标列无法转换为二值。")

def infer_time_cols(df):
    return [c for c in df.columns if re.search(r"(date|time|timestamp)", c, flags=re.I)]

def to_epoch(series):
    dt = pd.to_datetime(series, errors="coerce", utc=False)
    return dt.view("int64")/1e9

def sanitize_for_catboost(df, cat_cols):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == "object":
            s = df[c].astype(str)
            if s.str.upper().eq("NAT").any():
                df.loc[s.str.upper().eq("NAT"), c] = np.nan
    for c in df.columns:
        if c not in cat_cols and df[c].dtype == "object":
            num = pd.to_numeric(df[c], errors="coerce")
            if num.notna().sum() > 0:
                df[c] = num
        df[c] = df[c].replace({None: np.nan, np.inf: np.nan, -np.inf: np.nan})
    return df

def detect_amount_columns(df):
    cols = df.columns.str.lower().tolist()
    amount = balance = debit = credit = typcol = datecol = None
    for cand in ["amount","amt","txn_amt","transaction_amount","money","value","sum"]:
        if cand in cols: amount = df.columns[cols.index(cand)]; break
    for cand in ["balance","bal","acct_balance","running_balance"]:
        if cand in cols: balance = df.columns[cols.index(cand)]; break
    for cand in ["debit","debit_amount","dr_amt","out_amount"]:
        if cand in cols: debit = df.columns[cols.index(cand)]; break
    for cand in ["credit","credit_amount","cr_amt","in_amount"]:
        if cand in cols: credit = df.columns[cols.index(cand)]; break
    for cand in ["direction","type","txn_type","transaction_type","debit_credit"]:
        if cand in cols: typcol = df.columns[cols.index(cand)]; break
    for c in df.columns:
        if re.search(r"(date|time|timestamp)", c, flags=re.I):
            datecol = c; break
    return amount, balance, debit, credit, typcol, datecol

def build_statement_features(stmt, id_col, ref_ts):
    if stmt is None or stmt.empty:
        return pd.DataFrame(columns=[id_col])
    stmt = stmt.copy()
    amount, balance, debit, credit, typcol, datecol = detect_amount_columns(stmt)
    for c in [amount, balance, debit, credit]:
        if c is not None:
            stmt[c] = pd.to_numeric(stmt[c], errors="coerce")
    signed_amount = None
    if amount is not None and typcol is not None:
        typ = stmt[typcol]
        if pd.api.types.is_numeric_dtype(typ) and set(pd.unique(typ.dropna())) <= {0,1}:
            sign = np.where(typ==1, 1, -1)
        else:
            t = typ.astype(str).str.lower()
            sign = np.where(t.str.contains("debit|支出|转出|付款"), -1,
                            np.where(t.str.contains("credit|收入|转入|收款"), 1, 1))
        stmt["_signed_amount"] = stmt[amount] * sign
        signed_amount = "_signed_amount"
    elif debit is not None or credit is not None:
        stmt["_signed_amount"] = stmt.get(credit, 0).fillna(0) - stmt.get(debit, 0).fillna(0)
        signed_amount = "_signed_amount"

    if datecol is not None:
        stmt["_dt"] = pd.to_datetime(stmt[datecol], errors="coerce")
    else:
        stmt["_dt"] = pd.NaT

    agg = stmt.groupby(id_col).size().reset_index(name="stmt_txn_count")
    if signed_amount is not None:
        g = stmt.groupby(id_col)[signed_amount].agg(["sum","mean","std","min","max","median"]).add_prefix("stmt_amt_")
        agg = agg.merge(g, left_on=id_col, right_index=True, how="left")
    if amount is not None and typcol is not None:
        t = stmt[typcol]
        if not (pd.api.types.is_numeric_dtype(t) and set(pd.unique(t.dropna())) <= {0,1}):
            z = t.astype(str).str.lower()
            is_in = z.str.contains("credit|收入|转入|收款").astype(int)
        else:
            is_in = t.astype(int)
        stmt["_is_in"] = is_in
        gi = stmt[stmt["_is_in"]==1].groupby(id_col)[amount].agg(["sum","mean","std","min","max","median"]).add_prefix("stmt_in_")
        go = stmt[stmt["_is_in"]==0].groupby(id_col)[amount].agg(["sum","mean","std","min","max","median"]).add_prefix("stmt_out_")
        agg = agg.merge(gi, left_on=id_col, right_index=True, how="left").merge(go, left_on=id_col, right_index=True, how="left")
    if balance is not None:
        gb = stmt.groupby(id_col)[balance].agg(["mean","std","min","max","median"]).add_prefix("stmt_bal_")
        agg = agg.merge(gb, left_on=id_col, right_index=True, how="left")

    gtime = stmt.groupby(id_col)["_dt"].agg(["min","max"]).rename(columns={"min":"stmt_first_dt","max":"stmt_last_dt"})
    agg = agg.merge(gtime, left_on=id_col, right_index=True, how="left")
    for c in ["stmt_first_dt","stmt_last_dt"]:
        agg[c] = pd.to_datetime(agg[c], errors="coerce").view("int64")/1e9
    agg["stmt_active_days"] = (agg["stmt_last_dt"] - agg["stmt_first_dt"]) / 86400.0
    agg["stmt_days_since_first"] = (ref_ts - agg["stmt_first_dt"]) / 86400.0
    agg["stmt_days_since_last"]  = (ref_ts - agg["stmt_last_dt"]) / 86400.0
    agg["stmt_txn_intensity_per_day"] = agg["stmt_txn_count"] / (agg["stmt_active_days"].replace(0,np.nan) + 1.0)
    agg = agg.drop(columns=["stmt_first_dt","stmt_last_dt"])
    agg[id_col] = agg[id_col].astype(str)
    return agg

def winsorize_train_quantiles(X_train, X_other=None, lo=WINSOR_LO, hi=WINSOR_HI):
    X_train = X_train.copy()
    qmap = {}
    for c in X_train.columns:
        if pd.api.types.is_numeric_dtype(X_train[c]):
            ql = X_train[c].quantile(lo)
            qr = X_train[c].quantile(hi)
            qmap[c]=(ql,qr)
            X_train[c]=X_train[c].clip(lower=ql, upper=qr)
    if X_other is not None:
        X_other = X_other.copy()
        for c,(ql,qr) in qmap.items():
            if c in X_other.columns and pd.api.types.is_numeric_dtype(X_other[c]):
                X_other[c] = X_other[c].clip(lower=ql, upper=qr)
        return X_train, X_other, qmap
    return X_train, None, qmap

def _filter_valid_splits(X, y, splits):
    valid=[]
    for trn_idx, val_idx in splits:
        if len(trn_idx)==0 or len(val_idx)==0: continue
        y_tr=y.iloc[trn_idx]; y_val=y.iloc[val_idx]
        if y_tr.nunique()<2 or y_val.nunique()<2: continue
        valid.append((trn_idx, val_idx))
    return valid

def infer_time_col_for_cv(X):
    for c in X.columns:
        if c.startswith("days_since_") or c.startswith("stmt_days_since_"):
            return c
    for c in X.columns:
        if re.search(r"(date|time|timestamp)", c, flags=re.I):
            return c
    return None

def purged_time_series_split(X, y, n_splits=N_SPLITS, embargo_frac=EMBARGO_FRAC):
    tcol = infer_time_col_for_cv(X)
    if tcol is None: return None
    t = pd.to_numeric(X[tcol], errors="coerce")
    if t.notna().sum() < max(50, n_splits*10): return None
    order = np.argsort(t.fillna(t.min()-1).values)
    N=len(X); idx=np.arange(N)[order]
    base = N // (n_splits + 1)
    embargo = max(1, int(embargo_frac * N))
    splits=[]
    for k in range(1, n_splits+1):
        val_start = k*base
        val_end = (k+1)*base if k<n_splits else N
        if val_start>=N or val_end-val_start<=0: continue
        trn_right = max(0, val_start - embargo)
        trn_idx = idx[:trn_right]
        val_idx = idx[val_start:val_end]
        splits.append((trn_idx, val_idx))
    splits = _filter_valid_splits(X, y, splits)
    return splits if len(splits)>0 else None

def stratified_timebin_kfold(X, y, n_splits=N_SPLITS):
    tcol = infer_time_col_for_cv(X)
    if tcol is None: return None
    t = pd.to_numeric(X[tcol], errors="coerce")
    bins = pd.qcut(t, q=min(10, max(2, t.nunique())), labels=False, duplicates="drop")
    strat = (y.astype(int)*10 + bins.fillna(-1).astype(int)).astype(int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(skf.split(X, strat))
    splits = _filter_valid_splits(X, y, splits)
    return splits if len(splits)>0 else None

def build_splits(X, y):
    splits = purged_time_series_split(X, y)
    if splits is None: splits = stratified_timebin_kfold(X, y)
    if splits is None:
        for n in [5,4,3,2]:
            skf = StratifiedKFold(n_splits=n, shuffle=True, random_state=42)
            sp = list(skf.split(X, y))
            sp = _filter_valid_splits(X, y, sp)
            if len(sp)>0: splits = sp; break
    if splits is None:
        trn_idx, val_idx = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42, stratify=y)
        splits = _filter_valid_splits(X, y, [(trn_idx,val_idx)])
    return splits

def fit_cv_and_report(X, y, cat_cols, tag, seeds=SEEDS):
    splits = build_splits(X, y)
    if splits is None or len(splits)==0:
        raise RuntimeError("无法构造有效的CV折")

    pos_rate = y.mean()
    class_weights = [0.5/(1-pos_rate), 0.5/pos_rate] if 0<pos_rate<1 else None

    oof = np.zeros(len(X))
    models=[]; fi_rows=[]; cv_rows=[]

    for fold,(trn_idx,val_idx) in enumerate(splits,1):
        X_tr, X_val = X.iloc[trn_idx].copy(), X.iloc[val_idx].copy()
        y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]

        X_tr_w, X_val_w, qmap = winsorize_train_quantiles(X_tr, X_val, lo=WINSOR_LO, hi=WINSOR_HI)
        cat_eff = [c for c in cat_cols if (c in X_tr_w.columns and X_tr_w[c].dtype == "object")]
        train_pool = Pool(X_tr_w, y_tr, cat_features=[X_tr_w.columns.get_loc(c) for c in cat_eff])
        valid_pool = Pool(X_val_w, y_val, cat_features=[X_val_w.columns.get_loc(c) for c in cat_eff if c in X_val_w.columns])

        fold_pred = np.zeros(len(X_val_w))
        fold_models=[]
        for seed in seeds:
            model = CatBoostClassifier(
                iterations=2800, learning_rate=0.03, depth=6, l2_leaf_reg=16.0,  # 略加强正则
                loss_function="Logloss", eval_metric="AUC", random_seed=seed,
                od_type="Iter", od_wait=300, verbose=200,
                class_weights=class_weights, subsample=0.8, rsm=0.75,
                random_strength=1.0, bootstrap_type="Bernoulli"
            )
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=200)
            p = model.predict_proba(valid_pool)[:,1]
            fold_pred += p/len(seeds)
            fi = pd.DataFrame({"feature": X_tr_w.columns,
                               "importance": model.get_feature_importance(train_pool, type="FeatureImportance"),
                               "fold": fold, "seed": seed})
            fi_rows.append(fi)
            fold_models.append(model)

        oof[val_idx] = fold_pred
        auc = roc_auc_score(y_val, fold_pred) if y_val.nunique()>1 else np.nan
        tcol=None
        for c in X.columns:
            if c.startswith("days_since_") or c.startswith("stmt_days_since_"):
                tcol=c; break
        vmin=vmax=np.nan
        if tcol and tcol in X.columns:
            tt = pd.to_numeric(X.iloc[val_idx][tcol], errors="coerce")
            if tt.notna().any():
                vmin=float(np.nanmin(tt.values)); vmax=float(np.nanmax(tt.values))
        cv_rows.append({"fold":fold,"n_train":len(trn_idx),"n_valid":len(val_idx),
                        "pos_rate_train":float(y_tr.mean()),"pos_rate_valid":float(y_val.mean()),
                        "auc_valid":float(auc),"val_time_min":vmin,"val_time_max":vmax})
        models.append((fold_models, qmap, cat_eff))

    pd.DataFrame({"row_id":np.arange(len(oof)), "y_true":y, "oof_pred":oof}).to_csv(os.path.join(BASE,f"oof_v8a.csv"), index=False)
    cv_report = pd.DataFrame(cv_rows); cv_report.to_csv(os.path.join(BASE,f"cv_report_v8a.csv"), index=False)
    fi_all = pd.concat(fi_rows, axis=0)
    fi_all.to_csv(os.path.join(BASE,f"per_fold_importance_v8a.csv"), index=False)
    fi_mean = fi_all.groupby("feature", as_index=False)["importance"].mean().sort_values("importance", ascending=False)
    fi_mean.to_csv(os.path.join(BASE,f"feature_importance_v8a.csv"), index=False)

    # 校准十等分
    try:
        tmp = pd.DataFrame({"y":y, "score":oof}).dropna()
        tmp["decile"]=pd.qcut(tmp["score"], 10, labels=False, duplicates="drop")
        calib = tmp.groupby("decile").agg(y_rate=("y","mean"), count=("y","size"), score_mean=("score","mean")).reset_index().sort_values("decile", ascending=False)
        calib.to_csv(os.path.join(BASE, f"calib_bins_v8a.csv"), index=False)
    except Exception:
        pass

    return models, fi_mean

def main():
    train_main_path = 'train/train.csv'
    train_stmt_path = 'train/train_bank_statement.csv'
    test_main_path  = 'testaa/testaa.csv'
    test_stmt_path  = 'testaa/testaa_bank_statement.csv'

    train = load_csv_safe(train_main_path)
    stmt  = load_csv_safe(train_stmt_path)
    test  = load_csv_safe(test_main_path)
    stmt_t= load_csv_safe(test_stmt_path)

    assert train is not None, "缺少训练主表 train/train.csv"
    id_col = infer_id_col(train)
    tgt_col = infer_target_col(train)
    assert tgt_col is not None, "无法推断目标列，请手动指定。"

    leak_cols = [c for c in train.columns if re.search(r"(leak|target_leak)", c, flags=re.I)]
    if leak_cols: 
        train = train.drop(columns=leak_cols)
        if test is not None:
            test = test.drop(columns=[c for c in leak_cols if c in test.columns])

    # 参考时间（仅用训练）
    def max_ts_from(df):
        ts=[]
        for c in infer_time_cols(df):
            s = to_epoch(df[c])
            if s.notna().any(): ts.append(s.max())
        return np.nanmax(ts) if ts else None

    ref_ts = max([v for v in [max_ts_from(train), max_ts_from(stmt) if stmt is not None else None] if v is not None], default=None)

    # days_since_*（按训练识别出的时间列）
    def add_days_since(df, ref_ts, time_cols):
        if df is None or ref_ts is None or not time_cols: return df, []
        new_cols=[]
        for c in time_cols:
            s = to_epoch(df[c])
            col=f"days_since_{c}"
            df[col] = (ref_ts - s)/86400.0
            new_cols.append(col)
        df = df.drop(columns=[c for c in time_cols if c in df.columns])
        return df, new_cols

    tcols_tr = infer_time_cols(train)
    train, _ = add_days_since(train, ref_ts, tcols_tr)
    if test is not None:
        tcols_te = [c for c in test.columns if c in tcols_tr]
        test, _  = add_days_since(test, ref_ts, tcols_te)

    # 流水聚合
    if stmt is not None:
        stmt_id = infer_id_col(stmt)
        agg = build_statement_features(stmt, id_col=stmt_id, ref_ts=ref_ts if ref_ts is not None else 0.0)
        agg = agg.rename(columns={stmt_id:id_col})
        train[id_col]=train[id_col].astype(str); agg[id_col]=agg[id_col].astype(str)
        train = train.merge(agg, on=id_col, how="left", validate="m:1")
    if stmt_t is not None and test is not None:
        stmt_tid = infer_id_col(stmt_t)
        agg_t = build_statement_features(stmt_t, id_col=stmt_tid, ref_ts=ref_ts if ref_ts is not None else 0.0)
        agg_t = agg_t.rename(columns={stmt_tid:id_col})
        test[id_col]=test[id_col].astype(str); agg_t[id_col]=agg_t[id_col].astype(str)
        test = test.merge(agg_t, on=id_col, how="left", validate="m:1")

    # X / y
    y = ensure_binary_labels(train[tgt_col])
    X = train.drop(columns=[tgt_col]).copy()
    if id_col in X.columns: X = X.drop(columns=[id_col])
    if test is not None and id_col in test.columns:
        X_test = test.drop(columns=[id_col]).copy()
    else:
        X_test = test.copy() if test is not None else None

    # 自动&手工黑名单
    drops = []
    for fname in ["auto_drop_features_v8a.txt", "manual_drop_features.txt"]:
        p = os.path.join(BASE, fname)
        if os.path.exists(p):
            try:
                cols = [l.strip() for l in open(p, "r", encoding="utf-8").read().splitlines() if l.strip()]
                drops.extend(cols)
            except Exception:
                pass
    drops = sorted(list(set(drops)))
    if drops:
        X = X.drop(columns=[c for c in drops if c in X.columns], errors="ignore")
        if X_test is not None:
            X_test = X_test.drop(columns=[c for c in drops if c in X_test.columns], errors="ignore")
        pd.DataFrame({"dropped_feature": drops}).to_csv(os.path.join(BASE,"actually_dropped_v8a.csv"), index=False)

    # CatBoost 类别列
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]

    # winsorize + sanitize
    X, X_test, qmap = winsorize_train_quantiles(X, X_test, lo=WINSOR_LO, hi=WINSOR_HI)
    X = sanitize_for_catboost(X, cat_cols)
    if X_test is not None: X_test = sanitize_for_catboost(X_test, cat_cols)

    # 训练
    models, fi = fit_cv_and_report(X, y, cat_cols, tag="v8a", seeds=SEEDS)

    # 预测
    if X_test is not None:
        preds = np.zeros(len(X_test))
        for (fold_models, qmap_fold, cat_eff) in models:
            X_tw = X_test.copy()
            for c,(ql,qr) in qmap_fold.items():
                if c in X_tw.columns and pd.api.types.is_numeric_dtype(X_tw[c]):
                    X_tw[c] = X_tw[c].clip(lower=ql, upper=qr)
            X_tw = sanitize_for_catboost(X_tw, cat_eff)
            pool = Pool(X_tw, cat_features=[X_tw.columns.get_loc(c) for c in cat_eff if c in X_tw.columns and X_tw[c].dtype=="object"])
            fold_pred = np.mean([m.predict_proba(pool)[:,1] for m in fold_models], axis=0)
            preds += fold_pred / len(models)

        raw_id_col = id_col if id_col in test.columns else "id"
        ids = test[raw_id_col].astype(str).values if (test is not None and raw_id_col in test.columns) else np.arange(len(X_test))
        pd.DataFrame({str(raw_id_col): ids, "default_prob": preds}).to_csv(os.path.join(BASE,"submission_v8a.csv"), index=False)

if __name__ == "__main__":
    main()


0:	test: 0.6004046	best: 0.6004046 (0)	total: 32.9ms	remaining: 1m 31s
200:	test: 0.6349517	best: 0.6351214 (199)	total: 5.99s	remaining: 1m 17s
400:	test: 0.6379912	best: 0.6383768 (375)	total: 12s	remaining: 1m 11s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.638376781
bestIteration = 375

Shrink model to first 376 iterations.
0:	test: 0.5808567	best: 0.5808567 (0)	total: 25.1ms	remaining: 1m 10s
200:	test: 0.6367602	best: 0.6374275 (191)	total: 5.82s	remaining: 1m 15s
400:	test: 0.6343031	best: 0.6379099 (243)	total: 11.7s	remaining: 1m 9s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.6379099189
bestIteration = 243

Shrink model to first 244 iterations.
0:	test: 0.6294069	best: 0.6294069 (0)	total: 35.6ms	remaining: 1m 39s
200:	test: 0.6436788	best: 0.6436788 (200)	total: 7.61s	remaining: 1m 38s
400:	test: 0.6458945	best: 0.6463433 (396)	total: 15.3s	remaining: 1m 31s
600:	test: 0.6474738	best: 0.6491730 (497)	total: 23.1s	remaining: 1m